# Multi-label LP dataframe + 80:20 stratified split

Prepared 2026-09-24. Decisions taken in chat that morning:

| | Decision |
|---|---|
| Row set | all **3,377** images carrying at least one lens-presentation (LP) tag |
| Columns | the **5 LP tags only**. Object-of-Interest tags are left out (they are the YOLO line) |
| Split | **80:20**, **stratified** |
| Extra columns | none. Only information that adds value is retained |

This notebook builds the dataframe, loads the split, and proves the split is stratified.
It writes two files:

- `dataframe_3377_lp_labels.csv` - the label matrix, the format requested (`image_path` + 5 binary columns)
- `final_80_20_dataframe.csv` - the same rows with the split assignment attached, ready for train/test

**About the validation set.** She asked for 80:20. The split holds exactly 20% back for
final testing (`split_named` reads 80 / 20). Inside the 80%, 15% of the total is reserved
as a validation set so that early stopping can pick the best epoch. Without one, model
selection would have to run against the test set and the test number would come out
optimistic. The column `split` holds train/val/test; `split_named` holds her 80/20 view.


In [ ]:
from pathlib import Path
import csv
from collections import Counter

try:
    import pandas as pd
except ImportError:
    pd = None   # the checks below do not need pandas

LP  = ["P1_COC", "P1_FL", "P1_ML", "P1_MPP", "P1_NL"]
PRETTY = {
    "P1_COC": "Cavity_Off_Center",
    "P1_FL":  "Folded_Lens",
    "P1_ML":  "Multiple_Lenses",
    "P1_MPP": "Missing_Primary_Package",
    "P1_NL":  "Missing_Lens",
}

def find_dir():
    """Locate the experiment folder from wherever this notebook was opened."""
    here = Path.cwd()
    for p in [here, *here.parents]:
        for c in (p, p / "experiments" / "multilabel_resnet50"):
            if (c / "labels.csv").exists():
                return c
    raise SystemExit("labels.csv not found - open this notebook inside the repo")

D = find_dir()
print("working folder:", D)
print("pandas available:", pd is not None)

## 1. Load the label matrix and keep only LP-tagged images

`labels.csv` has one row for every image in the annotation export (6,097), with the 5 LP
columns plus provenance columns. Images carrying no LP tag are excluded: they are either
not annotated yet or tagged only with OOI conditions, neither of which belongs in this
classifier.


In [ ]:
rows = list(csv.DictReader(open(D / "labels.csv", newline="", encoding="utf-8")))
lp   = [r for r in rows if any(r[t] == "1" for t in LP)]

print("images in the export        :", len(rows))
print("images with at least 1 LP tag:", len(lp), "  <- rows kept")
print("images with 0 LP tags        :", len(rows) - len(lp), "  <- rows dropped")
print()
print("positives per column across the 3,377 kept rows")
for t in LP:
    n = sum(1 for r in lp if r[t] == "1")
    print("  %-26s %5d  (%.1f%%)" % (PRETTY[t], n, 100 * n / len(lp)))

assert len(lp) == 3377, "expected 3,377 LP-tagged images, got %d" % len(lp)

## 2. Write the dataframe in the requested format

One row per image, one binary column per LP tag. Nothing else is kept: the provenance
columns (`md5`, `task_id`, `task_name`, `n_tags`, `has_any_tag`, `negatives_reliable`)
are dropped from this file.

The **0s stay**. A 0 is not leftover metadata, it is the label itself: it says this image
was looked at and does not carry that condition. Those zeros are what the model learns
from, and they are also what `pos_weight` is computed from during training.


In [ ]:
header = ["image_path"] + [PRETTY[t] for t in LP]
matrix = []
for r in lp:
    matrix.append([r["image_path"]] + [int(r[t]) for t in LP])
matrix.sort(key=lambda x: x[0])

out = D / "dataframe_3377_lp_labels.csv"
with open(out, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(header)
    w.writerows(matrix)

print("wrote %s  (%d rows x %d columns)" % (out.name, len(matrix), len(header)))
print("columns:", header)
print()
if pd is not None:
    display(pd.DataFrame(matrix[:5], columns=header))
else:
    print(" | ".join(header))
    for row in matrix[:5]:
        print(" | ".join(str(v) for v in row))

## 3. Label patterns present

Five distinct combinations exist in the whole dataset. The only multi-label one is
Folded Lens + Multiple Lenses, and all 178 images carrying it are inside this file.


In [ ]:
patterns = Counter(tuple(row[1:]) for row in matrix)
print("distinct label combinations: %d\n" % len(patterns))
print("%-46s %6s" % ("combination", "rows"))
for pat, n in patterns.most_common():
    name = " + ".join(PRETTY[LP[i]] for i, v in enumerate(pat) if v) or "(none)"
    print("%-46s %6d" % (name, n))

multi = sum(n for pat, n in patterns.items() if sum(pat) > 1)
print("\nrows carrying 2 or more LP tags:", multi)
assert len(patterns) == 5 and multi == 178

## 4. Load the 80:20 split

Built with iterative multi-label stratification (Sechidis et al.) over the same 3,377
rows, seed 42, grouping on content md5 so identical pixels can never land in two splits.
Regenerate with `make_split_variants.py` if the annotation export is updated.


In [ ]:
split_path = D / "splits" / "variant_80_20.csv"
srows = list(csv.DictReader(open(split_path, newline="", encoding="utf-8")))
meta  = __import__("json").loads((D / "splits" / "variant_80_20.meta.json").read_text(encoding="utf-8"))

print("split file :", split_path.name)
print("rows       :", len(srows))
print("seed       :", meta["seed"], "| strategy:", meta["strategy"])
print("group leaks:", meta["group_split_leaks"], "(must be 0)")
print()
print("split column     :", dict(Counter(r["split"] for r in srows)))
print("split_named      :", dict(Counter(r["split_named"] for r in srows)))

assert len(srows) == 3377
assert len({r["image_path"] for r in srows}) == 3377, "duplicate image_path in split"
assert {r["image_path"] for r in srows} == {row[0] for row in matrix}, "split and dataframe disagree"
assert meta["group_split_leaks"] == 0

## 5. Proof that the split is stratified

For every LP column, compare the positives landing in each split against the number that
proportional allocation would give. `drift` is the relative error against that target.
Anything under a few percent means the class balance held.


In [ ]:
total_pos = {t: sum(int(r[t]) for r in lp) for t in LP}
splits = ["train", "val", "test"]
per = {s: {t: 0 for t in LP} for s in splits}
count = Counter()
for r in srows:
    count[r["split"]] += 1
    for t in LP:
        per[r["split"]][t] += int(r[t])

n = len(srows)
print("row counts (3-way, what training uses)")
for s in splits:
    print("  %-6s %5d  %6.2f%%" % (s, count[s], 100 * count[s] / n))
print("\nher framing (split_named)")
named = Counter(r["split_named"] for r in srows)
for s, c in named.most_common():
    print("  %-6s %5d  %6.2f%%" % (s, c, 100 * c / n))

print("\npositives per split, against the proportional target\n")
print("%-26s %6s | %-28s %-28s %-28s" % ("column", "all", "train", "val", "test"))
worst = 0.0
for t in LP:
    cells = []
    for s in splits:
        want = (count[s] / n) * total_pos[t]
        got  = per[s][t]
        drift = abs(got - want) / want if want else 0.0
        worst = max(worst, drift)
        cells.append("%4d / target %5.1f  (%+.1f%%)" % (got, want, 100 * (got - want) / want))
    print("%-26s %6d | %s" % (PRETTY[t], total_pos[t], "  ".join("%-28s" % c for c in cells)))

print("\nworst drift of any class in any split: %.2f%%" % (100 * worst))
assert worst < 0.05, "stratification drift too high: %.3f" % worst

## 6. Join the two files into one training-ready table


In [ ]:
where = {r["image_path"]: r for r in srows}
final = []
for row in matrix:
    s = where[row[0]]
    final.append(row + [s["split"], s["split_named"]])

fheader = header + ["split", "split_named"]
fout = D / "final_80_20_dataframe.csv"
with open(fout, "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(fheader)
    w.writerows(final)

print("wrote %s  (%d rows x %d columns)" % (fout.name, len(final), len(fheader)))
print("columns:", fheader)
print()
if pd is not None:
    display(pd.DataFrame(final[:5], columns=fheader))
else:
    print(" | ".join(fheader))
    for row in final[:5]:
        print(" | ".join(str(v) for v in row))

## 7. Checks

Every assertion below must pass before this goes for review.


In [ ]:
checks = []

checks.append(("row count is 3,377", len(matrix) == 3377))
checks.append(("no duplicate image_path", len({r[0] for r in matrix}) == len(matrix)))
checks.append(("every row carries at least one LP tag", all(sum(r[1:]) >= 1 for r in matrix)))
checks.append(("every label cell is 0 or 1", all(v in (0, 1) for r in matrix for v in r[1:])))
checks.append(("dataframe has 6 columns only", len(header) == 6))
checks.append(("split covers every row exactly once", len(final) == 3377))
checks.append(("split_named reads 80:20",
                abs(named["train"] / n - 0.80) < 0.01 and abs(named["test"] / n - 0.20) < 0.01))
checks.append(("validation set present and non-empty", count["val"] > 0))
checks.append(("no md5 group crosses a split", meta["group_split_leaks"] == 0))
checks.append(("all 178 multi-tag rows present",
                sum(1 for r in matrix if sum(r[1:]) == 2) == 178))
checks.append(("worst stratification drift under 5%", worst < 0.05))

for name, ok in checks:
    print("%s  %s" % ("PASS" if ok else "FAIL", name))

failed = [name for name, ok in checks if not ok]
print("\n%d/%d checks passed" % (len(checks) - len(failed), len(checks)))
assert not failed, "failed: %s" % failed

## 8. What this does and does not settle

**Settled:** row set (3,377), columns (5 LP tags, nothing else), split (80:20,
stratified, seed 42, verified above).

**Waiting on review before training:** the model recipe. ResNet50 with the classification
head replaced by `Linear(2048, 5)`, loss changed from categorical cross entropy to
binary cross entropy with logits (required, because one image can carry two tags and a
softmax cannot express that), targets as a float multi-hot vector, and evaluation by
per-class AUPRC instead of accuracy.

**Open points worth flagging:**

- Folded Lens appears only inside the Folded Lens + Multiple Lenses pair (178 of 178), so
  a model can score on it by copying the Multiple Lenses answer. Its AUPRC must be read
  against a random baseline of 0.053, and predictions checked by eye.
- The 20% test set holds 36 Folded Lens images, so recall there carries roughly
  plus or minus 10 points of uncertainty at 90% recall.
- When the completed annotation export arrives, rerun `build_labels.py` and
  `make_split_variants.py`, then rerun this notebook. No code changes needed.
